# 03 Mix Judger Policy

Notebook 02 searched module-wise G/A/S coefficients directly.  This notebook
turns those search traces into a reusable judger: a small model that scores
candidate mixtures from current round features and selects the best body/head/MoE
mix automatically.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '03_mix_judger_policy'
OUT


PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/03_mix_judger_policy')

## Train, Select, And Evaluate

The full evaluation is intentionally kept here because the goal is not only to
fit a predictor, but to verify whether its chosen coefficients behave like the
offline optimizer.


In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_03_train_mix_judger.py'),
    '--workspace-root', str(OUT),
    '--rounds', '1,2,3,4,5',
    '--pool-samples', '2200',
    '--observed-templates', '12',
    '--val-batch-size', '32',
    '--evaluate-full',
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_03_train_mix_judger.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/03_mix_judger_policy --rounds 1,2,3,4,5 --pool-samples 2200 --observed-templates 12 --val-batch-size 32 --evaluate-full --force


{
  "manifest": {
    "created_utc": "2026-05-13T13:04:22.563236+00:00",
    "protocol": "dqa_softmox_mix_judger_v1",
    "method": "learned score model over module-wise G/A/S candidate weights",
    "workspace": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/03_mix_judger_policy",
    "source_workspace": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full",
    "optimizer_workspaces": [
      "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer_expanded",
      "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/02_mix_weight_optimizer"
    ],
    "training_rows": 206,
    "full_training_rows": 19,
    "rounds": "1,2,3,4,5",
    "model_type": "extratrees"
  },
  "cv": [
    {
      "round": 1,
      "selected_candidate": "best00_sur01_00",
      "selected_score": 0.57245,
      "selected_pred_sc

CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_03_train_mix_judger.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/03_mix_judger_policy', '--rounds', '1,2,3,4,5', '--pool-samples', '2200', '--observed-templates', '12', '--val-batch-size', '32', '--evaluate-full', '--force'], returncode=0)

## Summaries


In [3]:
selected = pd.read_csv(OUT / 'stats' / '03_selected_weights.csv')
display(selected[['round','candidate_id','guard_reason','pred_score','body_g','body_a','body_s','head_g','head_a','head_s','moe_g','moe_a','moe_s','pool_size']])

eval_path = OUT / 'stats' / '03_selected_full_eval.csv'
if eval_path.exists():
    full_eval = pd.read_csv(eval_path)
    display(full_eval[['round','map50','map50_95','precision','recall','score','body_g','body_a','body_s','head_g','head_a','head_s','moe_g','moe_a','moe_s']])

cv = pd.read_csv(OUT / 'stats' / '03_leave_one_round_cv.csv')
display(cv)


,round,candidate_id,guard_reason,pred_score,body_g,body_a,body_s,head_g,head_a,head_s,moe_g,moe_a,moe_s,pool_size
0,1,prior02,NaN,0.57455,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,2101
1,2,prior07,NaN,0.57455,0.6500,0.2500,0.1000,0.2000,0.1000,0.7000,0.1500,0.7500,0.1000,2089
2,3,prior00,g_beats_children,0.57095,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,2089
3,4,prior00,repair_hurts_both,0.56565,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,2087
4,5,prior00,repair_hurts_both,0.55975,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,2090


,round,map50,map50_95,precision,recall,score,body_g,body_a,body_s,head_g,head_a,head_s,moe_g,moe_a,moe_s
0,1,0.462,0.260,0.694,0.431,0.57455,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998
1,2,0.462,0.260,0.694,0.431,0.57455,0.6500,0.2500,0.1000,0.2000,0.1000,0.7000,0.1500,0.7500,0.1000
2,3,0.459,0.258,0.684,0.433,0.57095,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001
3,4,0.455,0.255,0.690,0.428,0.56565,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001
4,5,0.450,0.253,0.686,0.424,0.55975,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001,0.9998,0.0001,0.0001


,best_candidate,best_map50,best_score,regret,round,selected_candidate,selected_map50,selected_pred_score,selected_score
0,best01_prior02,0.462,0.57455,0.00210,1,best00_sur01_00,0.461,0.570117,0.57245
1,best00_prior07,0.462,0.57455,0.00105,2,best01_sur00_02,0.461,0.569788,0.57350
2,best02_sur00_01,0.459,0.57100,0.00005,3,best00_prior00,0.459,0.569059,0.57095
3,best00_prior00,0.455,0.56565,0.00000,4,best02_sur00_01,0.455,0.563974,0.56565
4,best00_prior00,0.450,0.55975,0.00010,5,best02_prior01,0.450,0.568919,0.55965


In [4]:
print((OUT / '03_mix_judger_policy_report.md').read_text())


# DQA-SoftMoX Mix Judger Policy 03

- created_utc: 2026-05-13T13:04:22.563659+00:00
- training_rows: 206
- full_training_rows: 19
- model_type: extratrees

## Leave-One-Round Full-Candidate CV

| round | selected | best | selected score | best score | regret |
|---:|---|---|---:|---:|---:|
| 1 | best00_sur01_00 | best01_prior02 | 0.5725 | 0.5746 | 0.0021 |
| 2 | best01_sur00_02 | best00_prior07 | 0.5735 | 0.5746 | 0.0010 |
| 3 | best00_prior00 | best02_sur00_01 | 0.5710 | 0.5710 | 0.0000 |
| 4 | best02_sur00_01 | best00_prior00 | 0.5656 | 0.5656 | 0.0000 |
| 5 | best02_prior01 | best00_prior00 | 0.5596 | 0.5597 | 0.0001 |

## Selected Policy Weights

| round | pred | body G/A/S | head G/A/S | moe G/A/S | pool | guard |
|---:|---:|---|---|---|---:|---|
| 1 | 0.5745 | 0.00/0.00/1.00 | 0.00/0.00/1.00 | 0.00/0.00/1.00 | 2101 |  |
| 2 | 0.5745 | 0.65/0.25/0.10 | 0.20/0.10/0.70 | 0.15/0.75/0.10 | 2089 |  |
| 3 | 0.5709 | 1.00/0.00/0.00 | 1.00/0.00/0.00 | 1.00/0.00/0.00 | 2089 | g_beats_child